# SITS - Importar Dados Externos

Este notebook converte dados de fontes externas para o formato NPZ usado pelo SITS.

**Formato de entrada suportado:** `.aab` (raster tabular com labels e timeseries)

**Estrutura esperada do arquivo:**
- Coluna 0: Label (classe)
- Colunas 1-N: Metadados (ignorados)
- Colunas N+1 em diante: Timeseries (bandas intercaladas por timestep)

**Saida:** `*.npz` com arrays X (timeseries) e y (labels)

> **Nota:** Se voce ja tem dados no formato NPZ ou coletou amostras com o notebook 01, pode pular este notebook.

## 1. Configuracao

Edite os parametros abaixo conforme seu arquivo de entrada.

In [ ]:
from pathlib import Path
import numpy as np

# =============================================================================
# CONFIGURACAO - EDITE AQUI
# =============================================================================

# Arquivo de entrada (formato .aab ou similar)
INPUT_FILE = "../data/seu_arquivo.aab"

# Pasta de saida
OUTPUT_FOLDER = "../sessions/seu_projeto/annotation"

# Nome do arquivo de saida
OUTPUT_FILENAME = "dados_externos.npz"

# -----------------------------------------------------------------------------
# Parametros do dado
# -----------------------------------------------------------------------------
NUM_CHANNELS = 4      # Numero de bandas (ex: Blue, Green, Red, NIR)
NUM_TIMESTEPS = 12    # Numero de timesteps na serie temporal
BAND_NAMES = ["blue", "green", "red", "nir"]  # Nomes das bandas

# Estrutura das colunas do arquivo
LABEL_COL = 0         # Coluna do label (classe)
TS_START_COL = 7      # Coluna onde comeca a timeseries (0-indexed)

# -----------------------------------------------------------------------------
# Mapeamento de classes (valor no arquivo -> nome descritivo)
# -----------------------------------------------------------------------------
CLASS_NAMES = {
    1: "classe_1",
    2: "classe_2",
    3: "classe_3",
    # Adicione mais classes conforme necessario
}

# =============================================================================

print(f"Arquivo de entrada: {INPUT_FILE}")
print(f"Pasta de saida: {OUTPUT_FOLDER}")
print(f"Arquivo de saida: {OUTPUT_FILENAME}")

## 2. Carregar e Analisar Dados

In [ ]:
import rasterio

# Verificar se arquivo existe
input_path = Path(INPUT_FILE)
if not input_path.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {input_path.absolute()}")

# Carregar dados
with rasterio.open(input_path) as src:
    raster = src.read(1)  # Le primeira banda (dados em formato tabular)

print(f"Shape do raster: {raster.shape}")
print(f"Dtype: {raster.dtype}")
print(f"Numero de amostras: {raster.shape[0]}")
print(f"Numero de colunas: {raster.shape[1]}")

In [ ]:
# Analisar estrutura das colunas
print("Estrutura das colunas:")
print(f"  Coluna {LABEL_COL}: Labels (classes)")
print(f"  Colunas 1-{TS_START_COL-1}: Metadados (ignorados)")
print(f"  Colunas {TS_START_COL}+: Timeseries")
print()

# Verificar se o numero de colunas bate
expected_ts_cols = NUM_CHANNELS * NUM_TIMESTEPS
actual_ts_cols = raster.shape[1] - TS_START_COL

print(f"Colunas de timeseries esperadas: {expected_ts_cols} ({NUM_CHANNELS} bandas x {NUM_TIMESTEPS} timesteps)")
print(f"Colunas de timeseries encontradas: {actual_ts_cols}")

if actual_ts_cols != expected_ts_cols:
    print(f"\nAVISO: Numero de colunas nao bate!")
    print(f"Verifique NUM_CHANNELS, NUM_TIMESTEPS e TS_START_COL")
else:
    print("\nOK: Numero de colunas correto!")

In [ ]:
# Analisar distribuicao de classes
labels_raw = raster[:, LABEL_COL].astype(int)
unique_labels, counts = np.unique(labels_raw, return_counts=True)

print("Distribuicao de classes no arquivo:")
print("-" * 50)
total = 0
for label, count in zip(unique_labels, counts):
    class_name = CLASS_NAMES.get(label, f"classe_{label}")
    print(f"  {label}: {class_name:20s} - {count:6d} amostras")
    total += count
print("-" * 50)
print(f"  Total: {total:6d} amostras")
print(f"  Classes unicas: {len(unique_labels)}")

# Verificar se todas as classes tem nome
missing = [l for l in unique_labels if l not in CLASS_NAMES]
if missing:
    print(f"\nAVISO: Classes sem nome definido: {missing}")
    print("Adicione essas classes ao dicionario CLASS_NAMES")

## 3. Processar Dados

In [ ]:
def process_aab_data(raster, num_channels, num_timesteps, label_col, ts_start_col, class_names):
    """
    Processa dados do formato .aab para arrays numpy.
    
    Args:
        raster: Array 2D do raster (n_samples, n_cols)
        num_channels: Numero de bandas
        num_timesteps: Numero de timesteps
        label_col: Indice da coluna de labels
        ts_start_col: Indice da primeira coluna de timeseries
        class_names: Dict mapeando valor -> nome da classe
    
    Returns:
        X: Array (N, C, T) com timeseries
        y: Array (N,) com labels
        class_mapping: Dict nome -> indice
    """
    n_samples = raster.shape[0]
    
    # Extrair labels
    labels_raw = raster[:, label_col].astype(int)
    
    # Criar mapeamento de classes (ordenado por valor original)
    unique_labels = sorted(set(labels_raw))
    class_mapping = {}
    for idx, label in enumerate(unique_labels):
        name = class_names.get(label, f"classe_{label}")
        class_mapping[name] = idx
    
    # Mapear labels para indices
    label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
    y = np.array([label_to_idx[l] for l in labels_raw], dtype=np.int64)
    
    # Extrair timeseries
    ts_data = raster[:, ts_start_col:ts_start_col + num_channels * num_timesteps]
    
    # Reshape para (N, C, T)
    # Assumindo que os dados estao organizados como: B1T1, B1T2, ..., B1Tn, B2T1, B2T2, ...
    X = ts_data.reshape(n_samples, num_channels, num_timesteps).astype(np.float32)
    
    return X, y, class_mapping

# Processar
X, y, class_mapping = process_aab_data(
    raster=raster,
    num_channels=NUM_CHANNELS,
    num_timesteps=NUM_TIMESTEPS,
    label_col=LABEL_COL,
    ts_start_col=TS_START_COL,
    class_names=CLASS_NAMES,
)

print(f"X shape: {X.shape}  (N, C, T)")
print(f"y shape: {y.shape}  (N,)")
print(f"X dtype: {X.dtype}")
print(f"y dtype: {y.dtype}")
print()
print("Mapeamento de classes:")
for name, idx in sorted(class_mapping.items(), key=lambda x: x[1]):
    count = np.sum(y == idx)
    print(f"  {idx}: {name:20s} - {count:6d} amostras")

## 4. Verificar Dados

In [ ]:
# Estatisticas basicas
print("Estatisticas dos valores de X:")
print(f"  Min: {X.min():.4f}")
print(f"  Max: {X.max():.4f}")
print(f"  Mean: {X.mean():.4f}")
print(f"  Std: {X.std():.4f}")
print()

# Verificar valores invalidos
n_nan = np.isnan(X).sum()
n_inf = np.isinf(X).sum()
print(f"Valores NaN: {n_nan}")
print(f"Valores Inf: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("\nAVISO: Dados contem valores invalidos!")

In [ ]:
# Visualizar amostras
import matplotlib.pyplot as plt

n_classes = len(class_mapping)
n_cols = min(3, n_classes)
n_rows = (n_classes + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
if n_classes == 1:
    axes = [axes]
else:
    axes = axes.flatten() if n_classes > 1 else [axes]

idx_to_name = {v: k for k, v in class_mapping.items()}

for class_idx in range(n_classes):
    ax = axes[class_idx]
    
    # Primeira amostra desta classe
    sample_indices = np.where(y == class_idx)[0]
    if len(sample_indices) == 0:
        continue
    sample = X[sample_indices[0]]  # (C, T)
    
    # Plotar cada banda
    for band_idx, band_name in enumerate(BAND_NAMES):
        ax.plot(sample[band_idx], label=band_name, linewidth=1.5)
    
    ax.set_title(f"{idx_to_name[class_idx]}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Valor")
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

# Esconder eixos vazios
for i in range(n_classes, len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Exemplo de serie temporal por classe", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Salvar Dados

In [ ]:
import json
from datetime import datetime

# Criar pasta de saida
output_folder = Path(OUTPUT_FOLDER)
output_folder.mkdir(parents=True, exist_ok=True)

# Salvar NPZ
output_path = output_folder / OUTPUT_FILENAME
np.savez_compressed(output_path, X=X, y=y)

# Salvar metadados
idx_to_name = {v: k for k, v in class_mapping.items()}
metadata = {
    "source_file": input_path.name,
    "n_samples": int(X.shape[0]),
    "n_channels": int(X.shape[1]),
    "n_timesteps": int(X.shape[2]),
    "band_names": BAND_NAMES,
    "class_mapping": class_mapping,
    "idx_to_name": {str(k): v for k, v in idx_to_name.items()},
    "statistics": {idx_to_name[idx]: int(np.sum(y == idx)) for idx in range(len(class_mapping))},
    "created": datetime.now().isoformat(),
}

metadata_path = output_folder / OUTPUT_FILENAME.replace(".npz", "_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Mostrar resultado
npz_size = output_path.stat().st_size / 1024
json_size = metadata_path.stat().st_size / 1024

print(f"Arquivos salvos:")
print(f"  {output_path.name}: {npz_size:.1f} KB")
print(f"  {metadata_path.name}: {json_size:.1f} KB")
print()
print(f"Caminho: {output_folder.absolute()}")

## 6. Verificar Arquivo Salvo

In [ ]:
# Recarregar e verificar
data = np.load(output_path)
X_loaded = data["X"]
y_loaded = data["y"]

print("Dados recarregados:")
print(f"  X shape: {X_loaded.shape}")
print(f"  y shape: {y_loaded.shape}")
print()

# Verificar integridade
assert np.allclose(X, X_loaded), "X nao corresponde!"
assert np.array_equal(y, y_loaded), "y nao corresponde!"
print("Integridade verificada: OK")

In [ ]:
# Mostrar metadados
print("Metadados salvos:")
print("-" * 50)
with open(metadata_path, "r", encoding="utf-8") as f:
    meta = json.load(f)
    print(json.dumps(meta, indent=2, ensure_ascii=False))

---

## Resumo

Dados convertidos com sucesso!

**Arquivos gerados:**
- `*.npz`: Arrays numpy com X (timeseries) e y (labels)
- `*_metadata.json`: Metadados e mapeamento de classes

**Proximo passo:** 
- Se precisar combinar com outros dados, use o notebook **03_merge_dados.ipynb**
- Caso contrario, va direto para **04_treinamento.ipynb**